In [12]:
import json
import re
import shutil
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout

In [13]:
tf.keras.utils.set_random_seed(42)

In [14]:
PATH = "/content/Story.txt"

def clean_text(text):
    text = text.lower().replace("\u2018", "'").replace("\u2019", "'").replace("\u02bc", "'")
    return " ".join(re.findall(r"[a-z]+(?:'[a-z]+)?", text))

with open(PATH, "r", encoding="utf-8") as file:
    data = clean_text(file.read())

print("Story words:", len(data.split()))
print(data[:300])

Story words: 7542
section text three brothers were left destitute by the death of their parents at an early age the eldest was not yet able to provide fully for their support but he did all that he could in hunting with this aid and the stock of provisions already laid by in the lodge they managed to keep along they 


In [15]:
tokenizer = Tokenizer(oov_token="<UNK>", filters="", lower=False)
tokenizer.fit_on_texts([data])
total_words = len(tokenizer.word_index) + 1

tokens = tokenizer.texts_to_sequences([data])[0]
print("Vocabulary size:", total_words)
print("First 10 word numbers:", tokens[:10])

Vocabulary size: 1457
First 10 word numbers: [664, 665, 162, 74, 43, 184, 666, 46, 2, 318]


In [16]:
SEQ_LEN = 12
contexts, targets = [], []

for i in range(4, len(tokens)):
    for length in (4, 6, 8, 10, 12):
        contexts.append(tokens[max(0, i - length):i])
        targets.append(tokens[i])

x = pad_sequences(contexts, maxlen=SEQ_LEN, padding="pre")
y = np.array(targets, dtype=np.int32)

print("Inputs:", x.shape)
print("Answers:", y.shape)

Inputs: (37690, 12)
Answers: (37690,)


In [17]:
model = Sequential([
    Input(shape=(SEQ_LEN,), dtype="int32"),
    Embedding(total_words, 100),
    LSTM(256),
    Dropout(0.2),
    Dense(total_words, activation="softmax")
])

In [18]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [19]:
history = model.fit(x, y, epochs=20, batch_size=64)

Epoch 1/20
589/589 ━━━━━━━━━━━━━━━━━━━━ 22s 35ms/step - accuracy: 0.0745 - loss: 5.7486
Epoch 2/20
589/589 ━━━━━━━━━━━━━━━━━━━━ 21s 35ms/step - accuracy: 0.1237 - loss: 4.9635
Epoch 3/20
589/589 ━━━━━━━━━━━━━━━━━━━━ 21s 35ms/step - accuracy: 0.1785 - loss: 4.2046
Epoch 4/20
589/589 ━━━━━━━━━━━━━━━━━━━━ 21s 36ms/step - accuracy: 0.2536 - loss: 3.4886
Epoch 5/20
589/589 ━━━━━━━━━━━━━━━━━━━━ 21s 35ms/step - accuracy: 0.3708 - loss: 2.8045
Epoch 6/20
589/589 ━━━━━━━━━━━━━━━━━━━━ 21s 35ms/step - accuracy: 0.4977 - loss: 2.2028
Epoch 7/20
589/589 ━━━━━━━━━━━━━━━━━━━━ 21s 35ms/step - accuracy: 0.6113 - loss: 1.7158
Epoch 8/20
589/589 ━━━━━━━━━━━━━━━━━━━━ 21s 35ms/step - accuracy: 0.7037 - loss: 1.3264
Epoch 9/20
589/589 ━━━━━━━━━━━━━━━━━━━━ 21s 35ms/step - accuracy: 0.7816 - loss: 1.0211
Epoch 10/20
589/589 ━━━━━━━━━━━━━━━━━━━━ 21s 35ms/step - accuracy: 0.8365 - loss: 0.7861
Epoch 11/20
589/589 ━━━━━━━━━━━━━━━━━━━━ 21s 35ms/step - accuracy: 0.8806 - loss: 0.6026
Epoch 12/20
589/589 ━━━━━━━━━━

In [20]:
def predict_next_word(text):
    text = clean_text(text)
    tokens = tokenizer.texts_to_sequences([text])[0]
    unknown_id = tokenizer.word_index[tokenizer.oov_token]

    if not tokens or all(number == unknown_id for number in tokens[-SEQ_LEN:]):
        raise ValueError("Try words from the training story.")

    padded = pad_sequences([tokens], maxlen=SEQ_LEN, padding="pre", truncating="pre")
    probabilities = model(padded, training=False).numpy()[0]
    probabilities[0] = 0
    probabilities[unknown_id] = 0

    word_id = int(np.argmax(probabilities))
    return tokenizer.index_word[word_id]


text = "the silent kettle straightway served them"
print(text, "→", predict_next_word(text))


the silent kettle straightway served them → up


In [21]:
text = "Maidwa turned away, for it was very dark, and"
print(text, "→", predict_next_word(text))

Maidwa turned away, for it was very dark, and → it


In [22]:

model_folder = Path("model")
model_folder.mkdir(exist_ok=True)

model.save(model_folder / "next_word.keras")
with open(model_folder / "tokenizer.json", "w", encoding="utf-8") as file:
    file.write(tokenizer.to_json())

shutil.make_archive("model", "zip", root_dir=model_folder)
print("Saved next_word.keras and tokenizer.json. Download model.zip.")

Saved next_word.keras and tokenizer.json. Download model.zip.
